In [1]:
# Cell 1: Setup & Configuration
import os
import re
import io
import requests
import zipfile
import pandas as pd
import numpy as np
import logging
from datetime import datetime, timedelta
from pandas_datareader import data as pdr

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- PIPELINE CONFIGURATION ---

# 1. Date Range for the Pipeline
START_DATE = '2025-12-18' 
END_DATE   = '2025-12-20'

# 2. Target Stocks & Strict Aliases
# Logic: We ONLY keep news where the Title/URL explicitly mentions these keywords.
STOCK_CONFIG = {
    'AAPL': {
        'aliases': ['apple inc', 'apple computer', 'apple store', 'iphone', 'ipad', 'macbook', 'aapl'],
        'blacklist': ['apple valley', 'little apple', 'recipe', 'bake', 'pie', 'cider', 'yeast', 'orchard']
    },
    'AMZN': {
        'aliases': ['amazon.com', 'amazon inc', 'aws', 'prime video', 'jeff bezos', 'amzn'],
        'blacklist': ['amazon river', 'rainforest', 'fire', 'forest']
    },
    'TSLA': {
        'aliases': ['tesla', 'elon musk', 'spacex', 'cybertruck', 'tsla'],
        'blacklist': ['nikola tesla']
    },
    'MSFT': {
        'aliases': ['microsoft', 'azure', 'windows', 'satya nadella', 'msft'],
        'blacklist': []
    },
    'GOOGL': {
        'aliases': ['alphabet inc', 'google', 'youtube', 'sundar pichai', 'googl', 'goog'],
        'blacklist': []
    }
}

# 3. Layer Paths (Simulating S3/MinIO Buckets)
BASE_DIR = './data_pipeline'
BRONZE_DIR = f'{BASE_DIR}/bronze'   # Raw Data
SILVER_DIR = f'{BASE_DIR}/silver'   # Cleaned & Filtered
GOLD_DIR   = f'{BASE_DIR}/gold'     # Aggregated & ML Ready

for d in [BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    os.makedirs(d, exist_ok=True)
    
print("✓ Configuration Loaded & Directories Created")


✓ Configuration Loaded & Directories Created


In [2]:

# Cell 2: BRONZE LAYER - Ingestion (Download Only)
# Goal: Get data from the internet and save it "As-Is" to disk.

def download_gdelt_bronze(start, end, output_dir):
    """Downloads raw GKG zips and saves them as raw CSVs."""
    date_range = pd.date_range(start, end)
    
    for date_obj in date_range:
        date_str = date_obj.strftime('%Y%m%d')
        url = f"http://data.gdeltproject.org/gkg/{date_str}.gkg.csv.zip"
        save_path = f"{output_dir}/gdelt_gkg_{date_str}.csv"
        
        # Skip if already exists (Idempotency)
        if os.path.exists(save_path):
            logger.info(f"Skipping {date_str}, already exists.")
            continue
            
        logger.info(f"Downloading Bronze GDELT: {url}")
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                    csv_name = z.namelist()[0]
                    with z.open(csv_name) as f:
                        # We just dump the content to disk. No processing yet.
                        content = f.read()
                        with open(save_path, 'wb') as out:
                            out.write(content)
            else:
                logger.warning(f"GDELT not found for {date_str}")
        except Exception as e:
            logger.error(f"Failed to ingest {date_str}: {e}")

def download_stooq_bronze(tickers, output_dir):
    """Downloads raw stock history from Stooq."""
    for ticker in tickers:
        logger.info(f"Downloading Bronze Stooq: {ticker}")
        try:
            # Fetch long history to ensure we have moving averages
            df = pdr.DataReader(ticker, 'stooq', start=datetime(2020, 1, 1), end=datetime.now())
            if not df.empty:
                save_path = f"{output_dir}/stooq_{ticker}.csv"
                df.to_csv(save_path) # Saves raw Stooq output
        except Exception as e:
            logger.error(f"Stooq failed for {ticker}: {e}")

# --- EXECUTION ---
print("--- Starting Bronze Ingestion ---")
download_gdelt_bronze(START_DATE, END_DATE, BRONZE_DIR)
download_stooq_bronze(STOCK_CONFIG.keys(), BRONZE_DIR)
print("✓ Bronze Layer Complete")


2025-12-22 09:45:46,902 - INFO - Downloading Bronze GDELT: http://data.gdeltproject.org/gkg/20251218.gkg.csv.zip


--- Starting Bronze Ingestion ---


2025-12-22 09:45:54,737 - INFO - Downloading Bronze GDELT: http://data.gdeltproject.org/gkg/20251219.gkg.csv.zip
2025-12-22 09:46:02,005 - INFO - Downloading Bronze GDELT: http://data.gdeltproject.org/gkg/20251220.gkg.csv.zip
2025-12-22 09:46:06,623 - INFO - Downloading Bronze Stooq: AAPL
2025-12-22 09:46:10,269 - INFO - Downloading Bronze Stooq: AMZN
2025-12-22 09:46:11,943 - INFO - Downloading Bronze Stooq: TSLA
2025-12-22 09:46:13,583 - INFO - Downloading Bronze Stooq: MSFT
2025-12-22 09:46:19,210 - INFO - Downloading Bronze Stooq: GOOGL


✓ Bronze Layer Complete


In [ ]:
# Cell 3: SILVER LAYER - Transformation (Corrected with Word Boundaries)
# Goal: Read Bronze, Clean Title, Filter Strict Aliases (Distinct Words), Remove Noise.

def extract_clean_title(url):
    """Extracts title from URL slug."""
    if not isinstance(url, str): return ""
    try:
        # Regex: Grab text before .html or query params
        slug = re.search(r'([^/]+)(?:\.html|\?|$)', url.strip('/'))
        if slug:
            text = slug.group(1)
            # Remove digits, IDs, replace separators
            text = re.sub(r'\d+', '', text)
            text = text.replace('-', ' ').replace('_', ' ').replace('+', ' ')
            return text.strip().title()
    except:
        pass
    return ""

def process_gdelt_silver(bronze_dir, silver_dir, config):
    all_files = [f for f in os.listdir(bronze_dir) if f.startswith('gdelt_gkg_')]
    silver_rows = []
    
    for f in all_files:
        path = os.path.join(bronze_dir, f)
        try:
            df = pd.read_csv(path, sep='\t', header=None, on_bad_lines='skip', encoding='utf-8',
                             names=['DATE', 'NUMARTS', 'COUNTS', 'THEMES', 'LOCATIONS', 
                                    'PERSONS', 'ORGANIZATIONS', 'TONE', 'CAMEOEVENTIDS', 
                                    'SOURCES', 'SOURCEURLS'])
        except Exception as e:
            logger.warning(f"Could not read {f}: {e}")
            continue

        # 1. Extract Title
        df['extracted_title'] = df['SOURCEURLS'].apply(extract_clean_title)
        
        # 2. Loop Tickers & Filter
        for ticker, rules in config.items():
            aliases = rules['aliases']
            blacklist = rules['blacklist']
            
            # --- CORRECTED STRICT FILTER (Word Boundaries) ---
            # We wrap the alias pattern in \b (boundary) markers.
            # Example: \bapple\b matches "Apple" but NOT "Pineapple"
            # Example: \bipad\b matches "iPad" but NOT "Mipads"
            
            # Note: We iterate alias by alias to ensure safety with regex characters
            # Construct pattern: (\bword1\b|\bword2\b)
            boundary_aliases = [r'\b' + re.escape(a) + r'\b' for a in aliases]
            alias_pattern = '|'.join(boundary_aliases)
            
            mask_hit = df['extracted_title'].str.contains(alias_pattern, case=False, na=False, regex=True)
            
            # Blacklist Logic (Keep this simple/broad)
            if blacklist:
                blk_pattern = '|'.join([re.escape(b) for b in blacklist])
                mask_noise = df['SOURCEURLS'].astype(str).str.contains(blk_pattern, case=False, na=False)
                mask_hit = mask_hit & (~mask_noise)
                
            # Grab matches
            matches = df[mask_hit].copy()
            if not matches.empty:
                matches['ticker'] = ticker
                matches['date'] = pd.to_datetime(matches['DATE'].astype(str).str[:8], format='%Y%m%d')
                silver_rows.append(matches[['date', 'ticker', 'extracted_title', 'SOURCEURLS']])
                
    if silver_rows:
        df_silver = pd.concat(silver_rows, ignore_index=True)
        # Deduplicate
        df_silver = df_silver.drop_duplicates(subset=['date', 'ticker', 'extracted_title'])
        
        out_path = f"{silver_dir}/news_silver.csv"
        df_silver.to_csv(out_path, index=False)
        print(f"Saved Silver News: {out_path} ({len(df_silver)} rows)")
        return df_silver
    else:
        print("No relevant news found in Bronze.")
        return pd.DataFrame()

def process_stooq_silver(bronze_dir, silver_dir):
    all_files = [f for f in os.listdir(bronze_dir) if f.startswith('stooq_')]
    silver_stocks = []
    
    for f in all_files:
        ticker = f.replace('stooq_', '').replace('.csv', '')
        df = pd.read_csv(os.path.join(bronze_dir, f))
        
        # Stooq schema standardization
        df.columns = [c.lower() for c in df.columns]
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
            df['ticker'] = ticker
            silver_stocks.append(df[['date', 'ticker', 'close', 'open', 'high', 'low', 'volume']])
            
    if silver_stocks:
        df_stocks = pd.concat(silver_stocks, ignore_index=True)
        out_path = f"{silver_dir}/stocks_silver.csv"
        df_stocks.to_csv(out_path, index=False)
        print(f"Saved Silver Stocks: {out_path}")

# --- EXECUTION ---
print("--- Starting Silver Transformation ---")
df_news_silver = process_gdelt_silver(BRONZE_DIR, SILVER_DIR, STOCK_CONFIG)
process_stooq_silver(BRONZE_DIR, SILVER_DIR)
print("✓ Silver Layer Complete")

--- Starting Silver Transformation ---


NameError: name 'BRONZE_DIR' is not defined

In [4]:
# Cell 4: NLP Processing
# Goal: Prepare data for model, simulate scoring, and get ready for Gold aggregation.

def generate_nlp_input(silver_dir):
    """Creates the file your NLP model needs to read."""
    path = f"{silver_dir}/news_silver.csv"
    if not os.path.exists(path): return pd.DataFrame()
    
    df = pd.read_csv(path)
    # The NLP model needs strict columns: date, entity, text
    nlp_input = df[['date', 'ticker', 'extracted_title']].rename(columns={'ticker': 'entity', 'extracted_title': 'title'})
    
    # Save input for external model
    nlp_input.to_csv(f"{silver_dir}/nlp_input_ready.csv", index=False)
    print(f"Generated NLP Input: {silver_dir}/nlp_input_ready.csv")
    return nlp_input

def simulate_nlp_output(silver_dir):
    """
    MOCKS the output of your T5/BERT model. 
    In production, you would run the model script here instead.
    """
    input_df = generate_nlp_input(silver_dir)
    if input_df.empty: return pd.DataFrame()
    
    # Mocking a Sentiment Score (-1 to 1)
    # We add random noise to test the pipeline
    scored_df = input_df.copy()
    scored_df['sentiment_score'] = np.random.uniform(-1, 1, size=len(scored_df))
    
    # Save as if the model just finished running
    scored_df.to_csv(f"{silver_dir}/nlp_output_scored.csv", index=False)
    print(f"Simulated NLP Output: {silver_dir}/nlp_output_scored.csv")
    return scored_df

# --- EXECUTION ---
print("--- Starting NLP Processing ---")
df_scored_news = simulate_nlp_output(SILVER_DIR)


--- Starting NLP Processing ---
Generated NLP Input: ./data_pipeline/silver/nlp_input_ready.csv
Simulated NLP Output: ./data_pipeline/silver/nlp_output_scored.csv


In [5]:
# Cell 5: GOLD LAYER - Aggregation & ML Features
# Goal: Join Stocks + Sentiment, Create Lags, Target Variables.

def create_gold_layer(silver_dir, gold_dir):
    # 1. Load Data
    try:
        stocks = pd.read_csv(f"{silver_dir}/stocks_silver.csv")
        news = pd.read_csv(f"{silver_dir}/nlp_output_scored.csv")
        stocks['date'] = pd.to_datetime(stocks['date'])
        news['date'] = pd.to_datetime(news['date'])
    except Exception as e:
        logger.error(f"Missing Silver data: {e}")
        return

    # 2. Aggregate News (Many articles per day -> One score per day)
    # Logic: Weighted Average could be better, but we start with Mean.
    daily_sentiment = news.groupby(['date', 'entity']).agg({
        'sentiment_score': 'mean',
        'title': 'count'
    }).reset_index().rename(columns={'entity': 'ticker', 'sentiment_score': 'sentiment_avg', 'title': 'news_volume'})

    # 3. Merge (Left Join on Stocks to keep market days)
    gold_df = pd.merge(stocks, daily_sentiment, on=['date', 'ticker'], how='left')
    
    # Fill missing news with Neutral (0)
    gold_df['sentiment_avg'] = gold_df['sentiment_avg'].fillna(0)
    gold_df['news_volume'] = gold_df['news_volume'].fillna(0)

    # 4. Feature Engineering (The "Business Analytics" Value)
    gold_df = gold_df.sort_values(['ticker', 'date'])
    
    gold_dfs = []
    for ticker in gold_df['ticker'].unique():
        group = gold_df[gold_df['ticker'] == ticker].copy()
        
        # Lag 1: Yesterday's Sentiment predicts Today
        group['lag1_sentiment'] = group['sentiment_avg'].shift(1)
        
        # Moving Averages
        group['ma_7'] = group['close'].rolling(7).mean()
        group['ma_30'] = group['close'].rolling(30).mean()
        
        # Target: Next Day Return (for Prediction)
        group['daily_return'] = group['close'].pct_change()
        group['target_next_return'] = group['daily_return'].shift(-1)
        
        gold_dfs.append(group)
        
    final_gold = pd.concat(gold_dfs)
    
    # 5. Save Serving Tables
    # A. For Machine Learning (Drop rows with NaN targets)
    ml_data = final_gold.dropna()
    ml_data.to_csv(f"{gold_dir}/ml_training_data.csv", index=False)
    
    # B. For Dashboard (Keep everything to show gaps)
    final_gold.to_csv(f"{gold_dir}/superset_dashboard_data.csv", index=False)
    
    print(f"✓ Gold Layer Complete.")
    print(f"  - ML Data: {gold_dir}/ml_training_data.csv")
    print(f"  - Dashboard Data: {gold_dir}/superset_dashboard_data.csv")
    print("\nSample Data:")
    print(final_gold[['date', 'ticker', 'close', 'sentiment_avg', 'lag1_sentiment']].tail())

# --- EXECUTION ---
print("--- Starting Gold Aggregation ---")
create_gold_layer(SILVER_DIR, GOLD_DIR)

--- Starting Gold Aggregation ---
✓ Gold Layer Complete.
  - ML Data: ./data_pipeline/gold/ml_training_data.csv
  - Dashboard Data: ./data_pipeline/gold/superset_dashboard_data.csv

Sample Data:
           date ticker   close  sentiment_avg  lag1_sentiment
3006 2025-12-15   TSLA  475.31       0.000000        0.000000
3005 2025-12-16   TSLA  489.88       0.000000        0.000000
3004 2025-12-17   TSLA  467.26       0.000000        0.000000
3003 2025-12-18   TSLA  483.37      -0.071001        0.000000
3002 2025-12-19   TSLA  481.20       0.057264       -0.071001
